# Evolution Strategies

**Reinforcement Learning Course** — Notebook 05

| Algorithm | Environment | Key idea |
|-----------|-------------|----------|
| ES | InvertedDoublePendulum | Fitness-weighted gradient estimate |
| CMA-ES | InvertedDoublePendulum | Adaptive covariance search distribution |

In [ ]:
%pip install -q brax plotly

In [ ]:
import warnings
warnings.filterwarnings("ignore", message=".*Brax System.*not actively being maintained.*")

import numpy as np
import jax
import jax.numpy as jnp
import jax.flatten_util
import plotly.graph_objects as go
from brax import envs
from brax.io import html
from IPython.display import HTML

np.random.seed(42)
key = jax.random.PRNGKey(42)
print('devices:', jax.devices())

In [ ]:
env      = envs.get_environment('inverted_double_pendulum')
obs_dim  = env.observation_size
act_dim  = env.action_size
HIDDEN   = 8
MAX_STEPS = 500

# ── Policy: 2-layer MLP ───────────────────────────────────────────────────────
def policy_apply(params: dict, obs: jnp.ndarray) -> jnp.ndarray:
    x = jnp.tanh(obs @ params['W1'] + params['b1'])
    x = jnp.tanh(x  @ params['W2'] + params['b2'])
    return jnp.tanh(x @ params['W3'] + params['b3'])


key, k1, k2, k3 = jax.random.split(key, 4)
params0 = {
    'W1': jax.random.normal(k1, (obs_dim, HIDDEN)) * 0.1,
    'b1': jnp.zeros(HIDDEN),
    'W2': jax.random.normal(k2, (HIDDEN,  HIDDEN)) * 0.1,
    'b2': jnp.zeros(HIDDEN),
    'W3': jax.random.normal(k3, (HIDDEN,  act_dim)) * 0.1,
    'b3': jnp.zeros(act_dim),
}
theta0, unravel = jax.flatten_util.ravel_pytree(params0)
D = len(theta0)
print(f'obs_dim={obs_dim}  act_dim={act_dim}  D={D}')

# ── Rollout: fixed-length scan, reward masked after done ─────────────────────
@jax.jit
def evaluate(theta: jnp.ndarray, key: jax.Array) -> jnp.ndarray:
    params = unravel(theta)
    state  = env.reset(key)

    def step_fn(carry, _):
        state, acc_done = carry
        a = policy_apply(params, state.obs)
        next_state = env.step(state, a)
        r = jnp.where(acc_done, 0.0, next_state.reward)
        return (next_state, jnp.maximum(acc_done, next_state.done)), r

    (_, _), rewards = jax.lax.scan(
        step_fn, (state, jnp.float32(0.0)), None, length=MAX_STEPS
    )
    return rewards.sum()


@jax.jit
def evaluate_pop(thetas: jnp.ndarray, keys: jax.Array) -> jnp.ndarray:
    """Evaluate a population in parallel via vmap."""
    return jax.vmap(evaluate)(thetas, keys)

## Part 1 — Evolution Strategies

Estimate $\nabla_\theta J$ without backprop — sample $N$ perturbations $\epsilon_i \sim \mathcal{N}(0, I)$:

$$\nabla_\theta J(\theta) \approx \frac{1}{N\sigma}\sum_{i=1}^{N} F_i\,\epsilon_i, \qquad F_i = J(\theta + \sigma\epsilon_i)$$

**Fitness shaping:** normalise $F_i \leftarrow \frac{F_i - \bar{F}}{\text{std}(F)}$ to reduce variance and improve stability.

In [ ]:
N      = 50
SIGMA  = 0.05
LR_ES  = 0.02
N_ITER = 300

theta  = theta0.copy()
es_log: list[float] = []

for it in range(N_ITER):
    key, k1, k2 = jax.random.split(key, 3)
    eps  = jax.random.normal(k1, (N, D))
    keys = jax.random.split(k2, N)

    F      = evaluate_pop(theta + SIGMA * eps, keys)
    F_norm = (F - F.mean()) / (F.std() + 1e-8)
    theta  = theta + LR_ES * (eps * F_norm[:, None]).sum(0) / (N * SIGMA)

    es_log.append(float(F.mean()))
    if it % 50 == 0:
        print(f'ES  it {it:4d}  mean_fitness={es_log[-1]:.1f}')

print('Done.')

## Part 2 — CMA-ES

Adapts the search distribution $\mathcal{N}(m,\,\sigma^2 C)$. At each step, sample $\lambda$ candidates and select the top $\mu$.

**Weighted mean update:**
$$m \leftarrow \sum_{i=1}^{\mu} w_i\, x_{i:\lambda}$$

**Rank-1 + rank-$\mu$ covariance update:**
$$C \leftarrow (1-c_1-c_\mu)\,C + c_1\,p_c p_c^\top + c_\mu \sum_{i=1}^{\mu} w_i\, y_{i:\lambda} y_{i:\lambda}^\top$$

**Step-size** via cumulative evolution path $p_\sigma$:
$$\sigma \leftarrow \sigma\cdot\exp\!\left(\frac{c_\sigma}{d_\sigma}\!\left(\frac{\|p_\sigma\|}{\chi_N} - 1\right)\right)$$

In [ ]:
class CMAES:
    """(µ/µ_w, λ)-CMA-ES with rank-1 + rank-µ update and CSA step-size."""

    def __init__(self, D: int, lam: int = 50, sigma0: float = 0.3):
        self.D, self.lam = D, lam
        self.mu = lam // 2

        w = np.log(self.mu + 0.5) - np.log(np.arange(1, self.mu + 1))
        self.w       = w / w.sum()
        self.mu_eff  = 1.0 / (self.w ** 2).sum()

        self.cc    = (4 + self.mu_eff / D) / (D + 4 + 2 * self.mu_eff / D)
        self.cs    = (self.mu_eff + 2) / (D + self.mu_eff + 5)
        self.c1    = 2 / ((D + 1.3) ** 2 + self.mu_eff)
        self.cmu   = min(1 - self.c1,
                         2 * (self.mu_eff - 2 + 1/self.mu_eff) / ((D+2)**2 + self.mu_eff))
        self.damps = 1 + 2 * max(0, np.sqrt((self.mu_eff - 1) / (D + 1)) - 1) + self.cs
        self.chiN  = D**0.5 * (1 - 1/(4*D) + 1/(21*D**2))

        self.m      = np.zeros(D)
        self.sigma  = sigma0
        self.pc     = np.zeros(D)
        self.ps     = np.zeros(D)
        self.C      = np.eye(D)
        self.invsqC = np.eye(D)
        self._eval  = 0

    def ask(self) -> np.ndarray:
        L = np.linalg.cholesky(self.C + 1e-10 * np.eye(self.D))
        return self.m + self.sigma * (np.random.randn(self.lam, self.D) @ L.T)

    def tell(self, xs: np.ndarray, F: np.ndarray) -> None:
        self._eval += self.lam
        sel    = np.argsort(-F)[: self.mu]
        xs_top = xs[sel]
        m_old  = self.m.copy()
        self.m = (self.w[:, None] * xs_top).sum(0)
        step   = (self.m - m_old) / self.sigma

        self.ps = ((1 - self.cs) * self.ps
                   + np.sqrt(self.cs * (2 - self.cs) * self.mu_eff) * self.invsqC @ step)
        hsig = (np.linalg.norm(self.ps) / self.chiN
                / np.sqrt(1 - (1 - self.cs) ** (2 * self._eval / self.lam))) < 1.4 + 2 / (self.D + 1)

        self.pc = ((1 - self.cc) * self.pc
                   + hsig * np.sqrt(self.cc * (2 - self.cc) * self.mu_eff) * step)
        artmp  = (xs_top - m_old) / self.sigma
        self.C = ((1 - self.c1 - self.cmu) * self.C
                  + self.c1 * (np.outer(self.pc, self.pc)
                               + (1 - hsig) * self.cc * (2 - self.cc) * self.C)
                  + self.cmu * (self.w[:, None] * artmp).T @ artmp)
        self.sigma *= np.exp((self.cs / self.damps)
                             * (np.linalg.norm(self.ps) / self.chiN - 1))

        # Recompute C^{-1/2} periodically
        if self._eval % (10 * self.lam) == 0:
            self.C = np.triu(self.C) + np.triu(self.C, 1).T
            vals, vecs = np.linalg.eigh(self.C)
            self.invsqC = vecs @ np.diag(1 / np.sqrt(np.maximum(vals, 1e-20))) @ vecs.T

In [ ]:
cma = CMAES(D, lam=50, sigma0=0.3)
cma.m = np.array(theta0)
cma_log: list[float] = []

for it in range(N_ITER):
    xs   = cma.ask()                           # (lam, D) — numpy
    key, k1 = jax.random.split(key)
    keys = jax.random.split(k1, cma.lam)
    F    = np.array(evaluate_pop(jnp.array(xs), keys))
    cma.tell(xs, F)

    cma_log.append(F.mean())
    if it % 50 == 0:
        print(f'CMA it {it:4d}  mean_fitness={cma_log[-1]:.1f}')

print('Done.')

In [ ]:
def smooth(x: list[float], w: int = 10) -> np.ndarray:
    return np.convolve(x, np.ones(w) / w, mode='valid')

fig = go.Figure()
fig.add_trace(go.Scatter(
    y=es_log, mode='lines',
    line=dict(color='rgba(0,212,255,0.20)', width=1), name='ES raw',
))
fig.add_trace(go.Scatter(
    y=smooth(es_log), mode='lines',
    line=dict(color='rgba(0,212,255,1)', width=2), name='ES smoothed',
))
fig.add_trace(go.Scatter(
    y=cma_log, mode='lines',
    line=dict(color='rgba(245,158,11,0.20)', width=1), name='CMA-ES raw',
))
fig.add_trace(go.Scatter(
    y=smooth(cma_log), mode='lines',
    line=dict(color='rgba(245,158,11,1)', width=2), name='CMA-ES smoothed',
))
fig.update_layout(
    title='ES vs CMA-ES — InvertedDoublePendulum',
    xaxis_title='iteration', yaxis_title='mean fitness',
    template='plotly_dark', height=350,
)
fig.show()

In [ ]:
# Render best CMA-ES policy
# Tip: use mouse to rotate, scroll to zoom in the 3D viewer
@jax.jit
def eval_best(theta, key):
    params = unravel(theta)
    state = env.reset(key)
    def step_fn(carry, _):
        state = carry
        a = policy_apply(params, state.obs)
        next_state = env.step(state, a)
        return next_state, state.pipeline_state
    _, pipeline_states = jax.lax.scan(step_fn, state, None, length=MAX_STEPS)
    return pipeline_states

key, k_r = jax.random.split(key)
stacked = eval_best(jnp.array(cma.m), k_r)
states_list = [jax.tree.map(lambda x: x[i], stacked) for i in range(MAX_STEPS)]
HTML(html.render(env.sys, states_list, height=600))